# SAE on the leak-free model vs the leaked record (job 49729375)

**Question (hshunt):** *does it now learn something like the DD-algorithm, or at least think of relevant nodes and edges at the relevant times?*

**Answer:** partly, and the new part is the **edges**. Only the leak-free model isolates the *known* neighbours that force the answer — 16 `justifies_next` features against 2, best F1 **0.486 vs 0.162** at layer 8. Both models represent the node to emit at every layer and its neighbourhood from layer 8 on. The frontier does not separate them (~0.4 in both).

## Provenance

| | |
|---|---|
| Slurm job | `49729375` (`sud2saeLF`), COMPLETED 01:14:01, ExitCode 0:0 |
| git commit | `83714d7` — *Fix one-sided l1 search that invalidated SAE job 49702811* |
| sbatch | `scripts/slurm/sudoku2_sae_leakfree_vs_leaked.sbatch` |
| seed | `0` (SAE init + instance draw) |
| arm A, leak-free | `$SCRATCH/spin_gnn/sudoku_n24_leakfree_2026-10-26/d0256_h04_L12_ff01024_lr1e-03_pre_shuf` (job 49659502, shared-pool 0.8540) |
| arm B, leaked | `$OAK/hshunt/checkpoints/spin_gnn/sudoku_dd_d0256_L12_ntrain80000_ep300_sharedpool0.9960_job48912548` |
| SAE config | `--expansion 16` (d_sae=4096), `--l1-coeff 0.1`, `--l1-max-tries 8`, `--sae-epochs 1500`, `--alive-rate 1e-3`, `--n-null-shifts 19` |
| data | `--n-instances 3000`, `--max-positions-per-instance 40` → 53,200 positions, N=24 |
| layers | 0/2/4/6/8/10/12 at `--target-l0 64`; 8/12 repeated at 32 |
| out | `$SCRATCH/spin_gnn/sudoku_sae_2026-10-26_leakfree_l1fix/<arm>_l0_<target>/` |

The two checkpoints differ in **exactly one** training flag, `--shuffle-adj`. Each is analysed on **its own** instance distribution (`shuffle_adj` is threaded out of the checkpoint config), so neither is scored out of distribution.

## Why `--target-l0 64`

Measured on 3,468 positions from 200 N=24 instances: **55.8 of the 256 DD predicates are true simultaneously** per reasoning position (p10 46, p90 63). The previous target of 32 was below the ground truth's own complexity, so the dictionary could not carry the code even in principle.

## The run this supersedes

`49702811` ran the same script on the same checkpoints and exited 0:0 in 00:24:56. Its l1 search never searched: it only ever *raised* `l1_coeff` and accepted as soon as L0 fell *below* the target, so every one of the 18 sweeps stopped on iteration 1 at the start value, at L0 = 2–16 against a target of 64, and reported `hit_target_l0=True`. **Do not read `sudoku_sae_2026-10-26_leakfree/`** — its zero F1 on the edge families was a fact about the dictionary, not the model. The comparison below quantifies the damage.

In [ ]:
import ast, os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SCRATCH = os.environ["SCRATCH"]
NEW = f"{SCRATCH}/spin_gnn/sudoku_sae_2026-10-26_leakfree_l1fix"
VOID = f"{SCRATCH}/spin_gnn/sudoku_sae_2026-10-26_leakfree"

def load(arm, target=64, base=NEW):
    df = pd.read_csv(f"{base}/{arm}_l0_{target}/sae_summary.csv")
    df["fam"] = df.family_counts.map(ast.literal_eval)
    return df

lf, lk = load("leakfree"), load("leaked")
lf32, lk32 = load("leakfree", 32), load("leaked", 32)
void = load("leakfree", base=VOID)
len(lf), len(lk)

## 1. Metrics table

`best_*_f1` is the best F1 over all predicates in that family against the best-matching SAE feature; `median_best_f1_null` is the same statistic under a 19-roll position-shift null, which is the control that made the old "frontier F1 = 0.254" meaningless (on pure noise features the argmax alone gives 0.38–0.41).

In [ ]:
cols = ["layer", "l0", "hit_target_l0", "n_alive", "n_interpretable",
        "best_frontier_f1", "best_justifier_f1", "best_nbr_of_next_f1",
        "median_best_f1_null"]
tbl = pd.concat([lf[cols].assign(arm="A leak-free"), lk[cols].assign(arm="B leaked")])
tbl.set_index(["arm", "layer"]).round(3)

**Layer 0 is the one layer that misses the sparsity target in either arm** (L0 5.9 / 7.0 against 64), and that is not a failure of the search: it is the token+position embedding, whose true sparsity is ~2 at FVE = 1.000. The fixed search walks `l1` down through all 8 tries (0.1 → 6.1e-06), saturates, and prints the warning rather than claiming a hit. Read layer 0 as "the embedding is 2-sparse", not as a dictionary comparable to the others.

In [ ]:
fams = ["known", "color", "frontier", "justifies_next", "nbr_of_next",
        "next_node", "next_color", "remaining"]
rows = []
for arm, df in (("A leak-free", lf), ("B leaked", lk)):
    for _, r in df.iterrows():
        rows.append(dict(arm=arm, layer=int(r.layer),
                         **{f: r.fam.get(f, 0) for f in fams}))
counts = pd.DataFrame(rows).set_index(["arm", "layer"])
counts["book"] = counts.known + counts.color
counts["dd"] = counts.frontier + counts.justifies_next + counts.nbr_of_next
counts

## 2. The result: the forcing edges

`justifies_next[u]` = "u is a **known** neighbour of the node about to be emitted", i.e. one of the edges actually doing the forcing. `nbr_of_next[u]` = "u is a neighbour of that node", known or not.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, name) in zip(axes, [("best_justifier_f1", "justifies_next"),
                                  ("best_nbr_of_next_f1", "nbr_of_next"),
                                  ("best_frontier_f1", "frontier")]):
    ax.plot(lf.layer, lf[col], "o-", label="A leak-free (0.854)")
    ax.plot(lk.layer, lk[col], "o-", color="C3", label="B leaked (0.996)")
    ax.plot(lf.layer, lf.median_best_f1_null, "--", color="0.5", lw=1, label="null (A)")
    ax.set_xlabel("layer"); ax.set_ylabel(f"best {name} F1"); ax.set_ylim(0, 0.7)
    ax.set_title(name)
axes[0].legend(fontsize=8)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x, w = np.arange(len(lf)), 0.38
for ax, fam in zip(axes, ["justifies_next", "nbr_of_next"]):
    ax.bar(x - w/2, counts.loc["A leak-free", fam].values, w, label="A leak-free")
    ax.bar(x + w/2, counts.loc["B leaked", fam].values, w, color="C3", label="B leaked")
    ax.set_xticks(x); ax.set_xticklabels([int(v) for v in lf.layer])
    ax.set_xlabel("layer"); ax.set_ylabel("features"); ax.set_title(f"{fam} features (F1 >= 0.3)")
axes[0].legend()
fig.tight_layout()
counts.groupby("arm")[["justifies_next", "nbr_of_next", "frontier", "book", "next_node"]].sum()

## 3. Sparsity robustness: the same two layers at `--target-l0 32`

The family counts depend on a dictionary size *and* a sparsity level, so the headline ordering is re-checked at half the target.

In [ ]:
rob = pd.concat([
    lf32[cols].assign(arm="A leak-free", target=32), lk32[cols].assign(arm="B leaked", target=32),
    lf[lf.layer.isin([8, 12])][cols].assign(arm="A leak-free", target=64),
    lk[lk.layer.isin([8, 12])][cols].assign(arm="B leaked", target=64),
]).set_index(["target", "arm", "layer"]).sort_index()
rob[["l0", "n_interpretable", "best_justifier_f1", "best_frontier_f1",
     "best_nbr_of_next_f1"]].round(3)

A > B on `justifies_next` at both layers and both targets (0.245 vs 0.188 at L8, 0.252 vs 0.168 at L12), so the direction is not an artefact of the target. The *magnitude* is: the 3x gap is a target-64 phenomenon. `nbr_of_next` reverses at L8 under target 32 (0.211 vs 0.607) — at that sparsity the leaked model's neighbourhood feature survives and the leak-free model's does not, which is one more reason to treat single-family F1 values as structure rather than as an algorithm.

## 4. Comparison to the previous analysis of these same checkpoints

There is no earlier *valid* SAE run on the leak-free model, so the comparison with the previous best is twofold: arm **B is the previous best model** (0.9960), analysed here under the identical predicate scoring; and the void run below is the previous *attempt* at this exact comparison.

In [ ]:
cmp = pd.DataFrame({
    "layer": lf.layer.astype(int),
    "L0 void": void.l0.round(1).values, "L0 now": lf.l0.round(1).values,
    "n_alive void": void.n_alive.values, "n_alive now": lf.n_alive.values,
    "justif void": void.best_justifier_f1.round(3).values,
    "justif now": lf.best_justifier_f1.round(3).values,
    "nbr void": void.best_nbr_of_next_f1.round(3).values,
    "nbr now": lf.best_nbr_of_next_f1.round(3).values,
}).set_index("layer")
cmp

Every DD signal was understated by the void dictionaries, and `nbr_of_next` was reported as exactly 0.000 at four of seven layers where it is now 0.35–0.65. The fix is two-sided bracketing with log-space bisection, returning the fit closest to the target and flagging a miss in either direction, plus a regression test (`test_target_l0_search_backs_off_when_the_start_is_already_too_strong`) that starts the search deliberately in the collapse regime.

## 5. Pre-registrations, as resolved

Written into the sbatch header before the job ran.

| | claim | verdict | evidence |
|---|---|---|---|
| (a) | No layer-0 `next_node` features in A; B keeps them | **FALSIFIED** | 23 vs 24 features — identical. At `pre_color` positions `pos = i`, so the current token *is* `next_node`: a layer-0 token-identity feature matches the predicate for free, in both arms. |
| (b) | A has `justifies_next` features mid-stack, B fewer or none | **CONFIRMED** | 16 vs 2 features; best F1 0.486 vs 0.162 at layer 8; direction holds at target 32. |
| (c) | `best_frontier_f1` higher for A at every layer > 0 | **FALSIFIED** | Higher at 3 of 6 layers; both flat at 0.39–0.44. |
| (d) | Neither dictionary is DD-shaped; bookkeeping outnumbers frontier+justifies | **FALSIFIED**, backwards | Bookkeeping is nearly absent (0–19 features); `next_node` dominates every layer in both arms (55–218). |

(a), (c) and (d) were all wrong in the same direction: I expected the leak to be the *only* mechanism, so I expected the leaked model to look degenerate everywhere and the leak-free model to look DD-shaped everywhere. Neither happened. The one place they differ is the one place the leak made redundant — which known neighbours force the colour.

## 6. Caveats

- One seed, one checkpoint per arm, one instance distribution. Every number here is n = 1 in the only sense that matters for a mechanism claim.
- `best_f1` is a **max over 256 predicates**; the position-rolled null controls the argmax but does not make a single family's F1 a measure of the algorithm.
- The family counts use an F1 ≥ 0.3 cut and an aliveness floor of 1e-3.
- A 0.486 F1 on one family at one layer is structure, not DD. Nothing here shows the model *runs* the algorithm.

## 7. Owed next

1. Split the layer-0 `next_node` features by position `kind` — the predicate set already carries `kind_is_node`, so this is a scoring-side check, not a new run. If the matches are concentrated at `pre_color` positions, (a) was untestable as written rather than false.
2. Repeat on a second seed before this appears in a paper figure.